# 03 · Gold · `fila_inspecao`
### V-Energy Grid 360 — Arquitetura Medalhão

**Status: executada no Databricks, com todas as checagens `PASS` (32).** Fila de **906 UCs**: 32 nunca inspecionadas, 831 com irregularidade confirmada e consumo que não voltou, e 43 com a última inspeção sem irregularidade (rebaixadas). Conexão com a `gold.dim_uc` testada: 0 UCs fora da dimensão e 0 com município, bairro ou classe diferentes.

Responde ao pedido do Rafael: *"quero abrir o sistema e ver uma fila das unidades suspeitas, da mais promissora para a menos promissora, com o motivo da suspeita e quanta energia dá para recuperar em cada uma"*. Grava `v_energy.gold.fila_inspecao`, seguindo o **contrato da Gold (tabela B2, versão v4)**, com as 20 colunas do contrato.

**Grão:** 1 linha = 1 UC suspeita (as que passam do `SCORE_MIN` da regra). `prioridade` = 1 é a mais promissora.

**Fontes:** `gold.dim_uc` (consumo dos últimos 12 meses e dos 12 anteriores, medidor, GD, município e bairro) e `silver.inspecoes` (histórico de inspeções). Não lê a `fato_consumo_faturamento`: as médias de 12 meses já vêm na `dim_uc`.

## A regra `v1`

Toda a regra está nos **parâmetros da seção 2**; mudar um peso não muda as colunas, só a `versao_regra_score`. O `score_base` (0 a 100) soma sinais, e depois três ajustes usam o **histórico de inspeções**:

| Sinal | Pontos | Por quê |
|---|---|---|
| **Queda de consumo da própria UC** (média dos últimos 12 meses contra os 12 anteriores) | até **60**: 0 com queda de 10% ou menos, cheio com 40% ou mais | UCs sem GD com queda de 40% ou mais tiveram irregularidade confirmada em 91,7% (residencial) e 94,3% (comercial) das inspeções, contra ~8% nas estáveis (`analise_quatro_bairros`) |
| **Medidor eletromecânico** | **15** | 98,8% das UCs residenciais afetadas tinham medidor eletromecânico, contra ~23% na base |
| **Medidor com 15 anos ou mais** | **5** | a idade média das afetadas é de 15,9 anos (contra ~14) |
| **Consumo abaixo dos pares** (mediana de UCs sem GD da mesma classe, município e bairro) | até **15**: 0 com 20% abaixo, cheio com 60% abaixo | o sinal que o contrato pede; **não calibrado** (veja a seção 7) |
| **GD com energia injetada atípica** | **5** | o sinal da Silver de GD (log G5) |

| Ajuste | Efeito | Por quê |
|---|---|---|
| **Queda explicada por GD** | `score × 0,25` e **energia a recuperar = 0** | das 31 inspeções em UCs com GD, só 1 confirmou irregularidade (3,2%). A GD derruba o consumo em ~70% e, sem este ajuste, as 3.510 UCs com painel virariam suspeitas |
| **Última inspeção `SEM IRREGULARIDADE`** | `score × 0,5` | contrato: uma inspeção sem irregularidade rebaixa a UC |
| **Última inspeção `IRREGULARIDADE CONFIRMADA` e queda de 30% ou mais** | `+ 5` | contrato: a reincidência sobe a prioridade. No diagnóstico, o consumo das UCs confirmadas **não voltou** depois da inspeção |

Os pesos, os limiares e os fatores de ajuste são **decisões da regra v1, não estimativas estatísticas**. O que foi medido nas inspeções está na coluna "Por quê"; o resto é julgamento, e a seção 7 mede quão bem a regra separa as UCs que já foram inspecionadas.

**`kwh_estimado_recuperar_mes`:** `max(0, consumo_medio_12m_anterior_kwh − consumo_medio_12m_kwh)`, zero se a queda é explicada por GD (regra do contrato). **Subestima** as UCs cuja queda foi recente: a janela anterior de 12 meses já contém alguns meses depois da queda (nas UCs dos quatro bairros, a queda de maio e junho de 2025 entra na janela anterior).

**O que entra no dicionário de dados desta tabela:**
- `queda_consumo_pct` é **positivo** quando o consumo caiu (e 0 quando subiu); `consumo_vs_pares_pct` é **negativo** quando a UC consome menos que os pares;
- a `queda_explicada_por_gd` é verdadeira quando a UC tem GD conectada a partir de `2024-07-01` (início da janela anterior) e queda maior que zero; é uma aproximação da regra "a queda começou depois da conexão";
- os pares usam a mediana de **UCs sem GD** do segmento (classe, município e bairro) e só valem com 5 UCs ou mais; onde muitas UCs caíram juntas (os quatro bairros), a mediana dos pares também cai, e o sinal "abaixo dos pares" pode subestimar; **o efeito não foi medido**;
- a fila usa só as inspeções da **Silver**. As feitas no portal (ordens de inspeção, banco do backend) **não entram**;
- **a maioria das UCs com queda forte já foi inspecionada** (898 de 901): o topo da fila mostra, sobretudo, UCs com irregularidade confirmada cujo consumo **não voltou**, e só algumas nunca inspecionadas.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from decimal import Decimal
from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_dim = f"{gold_schema}.dim_uc"
tabela_insp = f"{silver_schema}.inspecoes"
tabela_stg = f"{gold_schema}.fila_inspecao_stg"
tabela_gold = f"{gold_schema}.fila_inspecao"
tabela_dq_log = f"{silver_schema}.dq_log"      # mesmo log das Silvers e das outras Golds

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_dim}, {tabela_insp}")
print(f"destino: {tabela_gold} (via {tabela_stg})")

origem:  v_energy.gold.dim_uc, v_energy.silver.inspecoes
destino: v_energy.gold.fila_inspecao (via v_energy.gold.fila_inspecao_stg)


## 2. Parâmetros da regra `v1`

Tudo o que a regra decide está aqui. Mudar um valor e rodar de novo gera outra versão da fila, e a **`VERSAO_REGRA` deve mudar junto**, para rastrear as mudanças de priorização.

- `QUEDA_MIN` e `QUEDA_PLENA`: a queda de consumo (%) em que o sinal começa a valer pontos e em que vale tudo. Com a janela de 12 meses, uma queda de 55% ocorrida em maio de 2025 aparece como ≈ 50%, e uma de 40%, como ≈ 36% (**estimativa nossa**: 10 dos 12 meses da janela anterior são anteriores à queda); por isso a rampa vai de 10% a 40%. A seção 7 mostra a taxa de acerto por faixa de queda, e é com ela que a rampa deve ser conferida.
- `SCORE_MIN`: abaixo disso, a UC não entra na fila. Com os pesos de `v1`, uma queda de 30% sozinha, sem outro sinal, já chega a 40.
- `QUEDA_PERSISTENTE`: a queda a partir da qual consideramos que "o consumo não voltou" (reincidência).

In [0]:
VERSAO_REGRA = "v1"

# Queda de consumo da própria UC (%, média de 12 meses contra os 12 anteriores)
QUEDA_MIN, QUEDA_PLENA = 10.0, 40.0
# Consumo abaixo dos pares (%, em valor absoluto): começa em PARES_MIN e vale tudo em PARES_PLENA
PARES_MIN, PARES_PLENA = 20.0, 60.0
PARES_MIN_UCS = 5                   # tamanho mínimo do segmento de pares
IDADE_ANTIGA = 15.0                 # medidor "antigo" (anos)

# Pesos do score_base (somam 100)
W_QUEDA, W_ELETROMEC, W_IDADE, W_PARES, W_ATIPICA = 60, 15, 5, 15, 5
assert W_QUEDA + W_ELETROMEC + W_IDADE + W_PARES + W_ATIPICA == 100

# Ajustes
FATOR_GD = 0.25                     # queda explicada por GD
FATOR_SEM_IRREGULARIDADE = 0.5      # última inspeção sem irregularidade
BONUS_REINCIDENCIA = 5              # última inspeção com irregularidade e o consumo não voltou
QUEDA_PERSISTENTE = 30.0

SCORE_MIN = 40                      # abaixo disso, a UC não entra na fila
INICIO_JANELA_ANTERIOR = "2024-07-01"   # GD conectada a partir daqui "explica" a queda entre as janelas

RESULTADO_CONFIRMADA = "IRREGULARIDADE CONFIRMADA"
RESULTADO_SEM = "SEM IRREGULARIDADE"
print(f"regra {VERSAO_REGRA}: score mínimo {SCORE_MIN}")

regra v1: score mínimo 40


## 2. Funções de checagem de qualidade

Mesmas funções da aula e das Silvers (`dq_check`, `dq_check_unique` e `dq_check_contagem`), com as mesmas colunas no log (`table_name`, `check_name`, `total_rows`, `failed_rows`, `passed`, `checked_at`). Acrescento `dq_check_valor`, para comparar um número calculado com um esperado (somas que precisam bater com as Silvers e com o diagnóstico); ela grava no log nas mesmas colunas, com `failed_rows` igual a 0 ou 1.

As funções só acumulam os resultados em `dq_results`. Depois das checagens, o notebook **grava o resultado em `silver.dq_log`** (sempre) e **só então decide se grava a Gold**.

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha. Quando o nulo também for um problema, há um `dq_check` separado com `isNotNull()`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado (soma que precisa bater com a Silver ou com o diagnóstico)."""
    passed = obtido == esperado
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
            failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 4. Leitura das fontes e conferência das colunas

Antes de construir, o notebook confere que a `gold.dim_uc` e a `silver.inspecoes` existem **com as colunas de que precisa**. Se faltar alguma, para aqui com uma mensagem clara.

Contagens esperadas: **45.000** UCs na `dim_uc` e **1.389** inspeções (871 com irregularidade confirmada, 62,7%).

In [0]:
dim = spark.table(tabela_dim)
insp = spark.table(tabela_insp)

exigidas = {
    tabela_dim: ["id_uc", "municipio", "bairro", "classe", "tipo_medidor_ativo", "idade_medidor_anos", "possui_gd", "data_conexao_gd",
                 "energia_injetada_atipica", "consumo_medio_12m_kwh", "consumo_medio_12m_anterior_kwh", "variacao_consumo_12m_pct",
                 "ultimo_mes_leitura"],
    tabela_insp: ["id_inspecao", "id_uc", "data_inspecao", "resultado", "irregularidade_confirmada"],
}
for tabela, colunas in exigidas.items():
    faltando = [c for c in colunas if c not in spark.table(tabela).columns]
    if faltando:
        raise Exception(f"{tabela} não tem as colunas {faltando} (DESCRIBE {tabela}).")

n_dim, n_insp = dim.count(), insp.count()
n_conf = insp.filter("irregularidade_confirmada").count()
print(f"UCs: {n_br(n_dim)} | inspeções: {n_br(n_insp)} | confirmadas: {n_br(n_conf)} ({n_conf / n_insp * 100:.1f}%)")   # esperado: 45.000 | 1.389 | 871 (62,7%)
print("resultados:", [r["resultado"] for r in insp.select("resultado").distinct().collect()])   # esperado: os 2 valores

UCs: 45.000 | inspeções: 1.389 | confirmadas: 871 (62.7%)
resultados: ['IRREGULARIDADE CONFIRMADA', 'SEM IRREGULARIDADE']


## 5. Sinais de cada UC

**Queda da própria UC:** `queda_consumo_pct = max(0, −variacao_consumo_12m_pct)`. Positiva quando o consumo caiu; nula se a UC não tem os 12 meses anteriores (61 UCs).

**Pares:** mediana do `consumo_medio_12m_kwh` das UCs **sem GD** da mesma classe, município e bairro, quando o segmento tem 5 UCs ou mais. `consumo_vs_pares_pct = (consumo da UC ÷ mediana − 1) × 100`: negativo quando a UC consome menos que os pares. A mediana inclui a própria UC; com 5 UCs ou mais, o efeito é pequeno.

**Queda explicada por GD:** a UC tem GD conectada em `2024-07-01` ou depois (a conexão está dentro da janela anterior ou depois dela) **e** queda maior que zero.

**Histórico de inspeções:** número de inspeções por UC e, da última (por data), o resultado e a data.

**Energia a recuperar:** `max(0, anterior − atual)`, zero se a queda é explicada por GD.

In [0]:
# --- pares: mediana das UCs sem GD do segmento
pares = (dim.filter(~F.col("possui_gd"))
         .groupBy("classe", "municipio", "bairro")
         .agg(F.count("*").alias("n_pares"),
              F.expr("percentile_approx(cast(consumo_medio_12m_kwh as double), 0.5)").alias("mediana_pares")))

# --- histórico de inspeções
w = Window.partitionBy("id_uc").orderBy(F.col("data_inspecao").desc(), F.col("id_inspecao").desc())
ultima = (insp.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
          .select("id_uc", F.col("resultado").alias("resultado_ultima_inspecao"), F.col("data_inspecao").alias("data_ultima_inspecao")))
contagem = insp.groupBy("id_uc").agg(F.count("*").alias("qtd_inspecoes_anteriores"))

def clamp01(x):
    return F.greatest(F.lit(0.0), F.least(F.lit(1.0), x))

fim_base = dim.agg(F.max("ultimo_mes_leitura").alias("m")).first()["m"]
print("último mês de leitura da base:", fim_base)     # esperado: 2026-06

sinais = (
    dim
    .join(pares, ["classe", "municipio", "bairro"], "left")
    .join(contagem, "id_uc", "left").join(ultima, "id_uc", "left")
    .withColumn("qtd_inspecoes_anteriores", F.coalesce("qtd_inspecoes_anteriores", F.lit(0)))
    .withColumn("queda_consumo_pct",
                F.when(F.col("variacao_consumo_12m_pct").isNotNull(),
                       F.round(F.greatest(F.lit(0.0), -F.col("variacao_consumo_12m_pct").cast("double")), 2)))
    .withColumn("consumo_vs_pares_pct",
                F.when((F.col("n_pares") >= PARES_MIN_UCS) & (F.col("mediana_pares") > 0),
                       F.round((F.col("consumo_medio_12m_kwh").cast("double") / F.col("mediana_pares") - 1) * 100, 2)))
    .withColumn("queda_explicada_por_gd",
                F.coalesce(F.col("possui_gd") & (F.coalesce("queda_consumo_pct", F.lit(0.0)) > 0)
                           & (F.col("data_conexao_gd") >= F.lit(INICIO_JANELA_ANTERIOR).cast("date")), F.lit(False)))
    .withColumn("kwh_estimado_recuperar_mes",
                F.when(F.col("queda_explicada_por_gd"), F.lit(0.0))
                 .otherwise(F.round(F.greatest(F.lit(0.0), (F.col("consumo_medio_12m_anterior_kwh") - F.col("consumo_medio_12m_kwh")).cast("double")), 2)))
    .withColumn("kwh_estimado_recuperar_mes", F.coalesce("kwh_estimado_recuperar_mes", F.lit(0.0)))
)
print("UCs com sinais calculados:", n_br(sinais.count()))               # esperado: 45.000
print("sem queda calculada (sem os 12 meses anteriores):", n_br(sinais.filter("queda_consumo_pct IS NULL").count()))   # esperado: 61
print("sem pares (segmento com menos de 5 UCs):", n_br(sinais.filter("consumo_vs_pares_pct IS NULL").count()))
print("queda explicada por GD:", n_br(sinais.filter("queda_explicada_por_gd").count()))

último mês de leitura da base: 2026-06
UCs com sinais calculados: 45.000
sem queda calculada (sem os 12 meses anteriores): 61
sem pares (segmento com menos de 5 UCs): 1.000
queda explicada por GD: 2.388


## 6. Score, motivo e prioridade

**`score_base`** soma os pontos dos sinais (seção 2). Os ajustes usam o histórico de inspeções (GD, `SEM IRREGULARIDADE`, reincidência), e o resultado é limitado entre 0 e 100.

**`motivo`:** texto montado só com os sinais que contribuem (queda, pares, medidor, GD atípica, e o histórico). É a coluna que o Rafael pediu: *o motivo da suspeita*.

**`prioridade`:** posição por `score` decrescente, depois por `kwh_estimado_recuperar_mes` decrescente e `id_uc` (para a ordem ser estável). Só entram as UCs com `score >= SCORE_MIN`.

**Por que a tabela passa por uma tabela temporária (`_stg`):** o Spark serverless não permite `.cache()`, e cada checagem recalcularia as junções e as janelas. A fila é gravada primeiro em `gold.fila_inspecao_stg`; as checagens leem dela; **só se todas passarem** a final é gravada e a temporária é apagada.

**Conferência:** a fila tem 20 colunas e as UCs com queda explicada por GD só aparecem nela se outro sinal as levar ao mínimo.

In [0]:
queda = F.coalesce("queda_consumo_pct", F.lit(0.0))
vs_pares = F.col("consumo_vs_pares_pct")
eletromec = (F.col("tipo_medidor_ativo") == "ELETROMECANICO")

pts_queda = W_QUEDA * clamp01((queda - QUEDA_MIN) / (QUEDA_PLENA - QUEDA_MIN))
pts_pares = F.when(vs_pares.isNotNull(), W_PARES * clamp01((-vs_pares - PARES_MIN) / (PARES_PLENA - PARES_MIN))).otherwise(F.lit(0.0))
pts_medidor = (F.when(eletromec, W_ELETROMEC).otherwise(0)
               + F.when(F.col("idade_medidor_anos").cast("double") >= IDADE_ANTIGA, W_IDADE).otherwise(0))
pts_atipica = F.when(F.col("possui_gd") & F.col("energia_injetada_atipica"), W_ATIPICA).otherwise(0)

score_base = pts_queda + pts_pares + pts_medidor + pts_atipica
ultima_res = F.col("resultado_ultima_inspecao")
fator = (F.when(F.col("queda_explicada_por_gd"), FATOR_GD).otherwise(1.0)
         * F.when(ultima_res == RESULTADO_SEM, FATOR_SEM_IRREGULARIDADE).otherwise(1.0))
bonus = F.when((ultima_res == RESULTADO_CONFIRMADA) & (queda >= QUEDA_PERSISTENTE), BONUS_REINCIDENCIA).otherwise(0)

mes = lambda c: F.date_format(c, "yyyy-MM")
motivo = F.concat_ws("; ",
    F.when(queda >= 15, F.concat(F.lit("consumo caiu "), F.round(queda, 0).cast("int").cast("string"), F.lit("% vs. 12 meses anteriores"))),
    F.when(vs_pares <= -30, F.concat(F.lit("consumo "), F.round(-vs_pares, 0).cast("int").cast("string"), F.lit("% abaixo dos pares do bairro"))),
    F.when(eletromec, F.concat(F.lit("medidor eletromecânico com "), F.round(F.col("idade_medidor_anos").cast("double"), 0).cast("int").cast("string"), F.lit(" anos")))
     .when(F.col("idade_medidor_anos").cast("double") >= IDADE_ANTIGA, F.concat(F.lit("medidor com "), F.round(F.col("idade_medidor_anos").cast("double"), 0).cast("int").cast("string"), F.lit(" anos"))),
    F.when(F.col("possui_gd") & F.col("energia_injetada_atipica"), F.lit("energia injetada pela GD atípica")),
    F.when(F.col("queda_explicada_por_gd"), F.lit("queda explicada pela geração distribuída (rebaixada)")),
    F.when(ultima_res == RESULTADO_CONFIRMADA,
           F.concat(F.lit("irregularidade confirmada em "), mes("data_ultima_inspecao"),
                    F.when(queda >= QUEDA_PERSISTENTE, F.lit(" e o consumo não voltou")).otherwise(F.lit("")))),
    F.when(ultima_res == RESULTADO_SEM, F.concat(F.lit("última inspeção ("), mes("data_ultima_inspecao"), F.lit(") sem irregularidade (rebaixada)"))),
)

pontuada = (
    sinais
    .withColumn("score_base", F.round(score_base, 1))
    .withColumn("score", F.round(F.greatest(F.lit(0.0), F.least(F.lit(100.0), score_base * fator + bonus)), 1))
    .withColumn("motivo", F.coalesce(F.when(motivo != "", motivo), F.lit("sem sinal dominante")))
)

fila_base = pontuada.filter(F.col("score") >= SCORE_MIN)
w_pri = Window.orderBy(F.col("score").desc(), F.col("kwh_estimado_recuperar_mes").desc(), F.col("id_uc"))

df_fila = (
    fila_base
    .withColumn("prioridade", F.row_number().over(w_pri))
    .select(
        "id_uc", "prioridade",
        F.col("score").cast("decimal(5,1)").alias("score"),
        "motivo",
        F.col("kwh_estimado_recuperar_mes").cast("decimal(10,2)").alias("kwh_estimado_recuperar_mes"),
        F.col("queda_consumo_pct").cast("decimal(6,2)").alias("queda_consumo_pct"),
        F.col("consumo_vs_pares_pct").cast("decimal(7,2)").alias("consumo_vs_pares_pct"),
        F.col("tipo_medidor_ativo").alias("tipo_medidor"),
        F.col("idade_medidor_anos").cast("decimal(5,1)").alias("idade_medidor_anos"),
        "possui_gd", "queda_explicada_por_gd", "energia_injetada_atipica",
        "qtd_inspecoes_anteriores", "resultado_ultima_inspecao", "data_ultima_inspecao",
        "municipio", "bairro", "classe",
        F.last_day(F.to_date(F.concat(F.lit(fim_base), F.lit("-01")))).alias("data_referencia_score"),
        F.lit(VERSAO_REGRA).alias("versao_regra_score"),
    )
)
print("colunas:", len(df_fila.columns))   # esperado: 20

# Grava na tabela temporária; as checagens leem dela (sem recalcular as junções e as janelas)
(
    df_fila.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_stg)
)
fila = spark.table(tabela_stg)
print("UCs na fila (score >= %d): %s" % (SCORE_MIN, n_br(fila.count())))
display(fila.orderBy("prioridade").limit(15))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


colunas: 20
UCs na fila (score >= 40): 906


id_uc,prioridade,score,motivo,kwh_estimado_recuperar_mes,queda_consumo_pct,consumo_vs_pares_pct,tipo_medidor,idade_medidor_anos,possui_gd,queda_explicada_por_gd,energia_injetada_atipica,qtd_inspecoes_anteriores,resultado_ultima_inspecao,data_ultima_inspecao,municipio,bairro,classe,data_referencia_score,versao_regra_score
UC101238,1,100.0,consumo caiu 58% vs. 12 meses anteriores; consumo 68% abaixo dos pares do bairro; medidor eletromecânico com 16 anos; irregularidade confirmada em 2025-12 e o consumo não voltou,175.92,57.79,-67.83,ELETROMECANICO,15.9,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-12-03,Jaboatão dos Guararapes,Prazeres,COMERCIAL,2026-06-30,v1
UC102335,2,100.0,consumo caiu 52% vs. 12 meses anteriores; consumo 61% abaixo dos pares do bairro; medidor eletromecânico com 18 anos; irregularidade confirmada em 2025-09 e o consumo não voltou,171.83,52.14,-60.51,ELETROMECANICO,18.0,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-09-26,Jaboatão dos Guararapes,Prazeres,COMERCIAL,2026-06-30,v1
UC105160,3,100.0,consumo caiu 56% vs. 12 meses anteriores; consumo 67% abaixo dos pares do bairro; medidor eletromecânico com 26 anos; irregularidade confirmada em 2025-12 e o consumo não voltou,169.08,55.83,-66.51,ELETROMECANICO,25.9,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-12-29,Jaboatão dos Guararapes,Prazeres,COMERCIAL,2026-06-30,v1
UC100203,4,100.0,consumo caiu 57% vs. 12 meses anteriores; consumo 66% abaixo dos pares do bairro; medidor eletromecânico com 20 anos; irregularidade confirmada em 2025-08 e o consumo não voltou,148.58,57.15,-66.36,ELETROMECANICO,19.7,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-08-30,Recife,Macaxeira,COMERCIAL,2026-06-30,v1
UC124391,5,100.0,consumo caiu 53% vs. 12 meses anteriores; consumo 67% abaixo dos pares do bairro; medidor eletromecânico com 22 anos; irregularidade confirmada em 2025-12 e o consumo não voltou,127.41,53.46,-66.56,ELETROMECANICO,21.8,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-12-30,Jaboatão dos Guararapes,Jardim Jordão,COMERCIAL,2026-06-30,v1
UC139423,6,100.0,consumo caiu 50% vs. 12 meses anteriores; consumo 70% abaixo dos pares do bairro; medidor eletromecânico com 20 anos; irregularidade confirmada em 2025-09 e o consumo não voltou,114.25,50.24,-70.28,ELETROMECANICO,19.9,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-09-10,Recife,Iputinga,COMERCIAL,2026-06-30,v1
UC113436,7,100.0,consumo caiu 56% vs. 12 meses anteriores; consumo 78% abaixo dos pares do bairro; medidor eletromecânico com 17 anos; irregularidade confirmada em 2025-12 e o consumo não voltou,103.33,55.68,-78.40,ELETROMECANICO,17.3,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-12-25,Recife,Iputinga,COMERCIAL,2026-06-30,v1
UC128921,8,100.0,consumo caiu 47% vs. 12 meses anteriores; consumo 67% abaixo dos pares do bairro; medidor eletromecânico com 25 anos; irregularidade confirmada em 2025-10 e o consumo não voltou,95.42,46.54,-66.96,ELETROMECANICO,25.3,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-10-27,Jaboatão dos Guararapes,Jardim Jordão,COMERCIAL,2026-06-30,v1
UC133384,9,100.0,consumo caiu 50% vs. 12 meses anteriores; consumo 77% abaixo dos pares do bairro; medidor eletromecânico com 16 anos; irregularidade confirmada em 2025-09 e o consumo não voltou,94.34,50.47,-76.82,ELETROMECANICO,15.6,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-09-26,Jaboatão dos Guararapes,Prazeres,COMERCIAL,2026-06-30,v1
UC123759,10,100.0,consumo caiu 47% vs. 12 meses anteriores; consumo 70% abaixo dos pares do bairro; medidor eletromecânico com 15 anos; irregularidade confirmada em 2025-09 e o consumo não voltou,90.50,47.24,-69.52,ELETROMECANICO,15.2,false,false,false,1,IRREGULARIDADE CONFIRMADA,2025-09-05,Jaboatão dos Guararapes,Jardim Jordão,COMERCIAL,2026-06-30,v1


## 7. Validação: a regra separa as UCs que já foram inspecionadas?

O contrato pede: *as UCs no topo da fila que já foram inspecionadas devem ter taxa de acerto acima de 62,7%*, a taxa histórica. Aqui a regra é testada contra as 1.389 inspeções, usando o `score_base`, que **não usa o histórico de inspeções**, para o teste não ser circular.

**Limites:**
- é uma validação **retrospectiva**: o consumo dos 12 últimos meses inclui meses posteriores às inspeções (agosto a dezembro de 2025), embora a queda já existisse antes;
- os pesos foram escolhidos pela evidência da `analise_quatro_bairros`, que usa as **mesmas** inspeções: a taxa de acerto das faixas de cima é, em parte, **conhecida de antemão**. Serve para checar se a implementação reproduz o que sabemos e para ajustar os limiares, mas **não é validação independente**;
- 898 das 901 UCs com queda forte já foram inspecionadas, então quase não há UCs "ainda não inspecionadas" para testar a regra com dados novos.

**Esperado:** a faixa de 80 a 100 do `score_base` com taxa de acerto próxima de 90% e as faixas baixas, perto de 8%. Os números de cada faixa e a de `queda_consumo_pct` mostram onde ajustar a rampa da queda.

**Resultado da execução:**

| Faixa do `score_base` | Inspeções | Confirmadas | Taxa |
|---|---|---|---|
| 0 a 20 | 400 | 35 | 8,8% |
| 20 a 40 | 77 | 5 | 6,5% |
| 40 a 60 | 3 | 0 | 0% |
| 60 a 80 | 344 | 311 | 90,4% |
| 80 a 100 | 565 | 520 | 92,0% |

Pontuação de 60 ou mais: **909 inspeções, 831 confirmadas (91,4%)**, contra **8,4%** abaixo de 40 (40 de 477), e acima dos 62,7% históricos. Por queda de consumo (UCs sem GD): menos de 10%, 8,1% (456 inspeções); de 30% a 40%, 90,0% (20); de 40% a 50%, 92,1% (404); de 50% ou mais, 93,0% (474). **Não há inspeções com queda entre 10% e 30%** (só 1): os dados são bimodais, e a parte da rampa entre 10% e 30% **não pôde ser testada**. UCs com GD: 31 inspeções e 1 confirmada (3,2%).

In [0]:
insp_uc = (insp.select("id_uc", "irregularidade_confirmada")
           .join(pontuada.select("id_uc", "score_base", "score", "queda_consumo_pct", "possui_gd", "queda_explicada_por_gd"), "id_uc", "left"))
print("inspeções com UC na dim_uc:", n_br(insp_uc.filter("score_base IS NOT NULL").count()), "de", n_br(n_insp))   # esperado: 1.389 de 1.389

def taxa(df, coluna):
    return (df.groupBy(coluna).agg(F.count("*").alias("inspecoes"),
                                   F.sum(F.col("irregularidade_confirmada").cast("int")).alias("confirmadas"),
                                   F.round(F.avg(F.col("irregularidade_confirmada").cast("int")) * 100, 1).alias("pct_confirmadas"))
            .orderBy(coluna))

faixa_score = (F.when(F.col("score_base") >= 80, "e) 80 a 100").when(F.col("score_base") >= 60, "d) 60 a 80")
                .when(F.col("score_base") >= 40, "c) 40 a 60").when(F.col("score_base") >= 20, "b) 20 a 40").otherwise("a) 0 a 20"))
print("taxa de acerto por faixa do score_base (referência: 62,7% no total):")
display(taxa(insp_uc.withColumn("faixa_score", faixa_score), "faixa_score"))

faixa_queda = (F.when(F.col("queda_consumo_pct").isNull(), "z) sem os 12 meses anteriores")
                .when(F.col("queda_consumo_pct") >= 50, "f) queda 50% ou mais").when(F.col("queda_consumo_pct") >= 40, "e) queda 40% a 50%")
                .when(F.col("queda_consumo_pct") >= 30, "d) queda 30% a 40%").when(F.col("queda_consumo_pct") >= 20, "c) queda 20% a 30%")
                .when(F.col("queda_consumo_pct") >= 10, "b) queda 10% a 20%").otherwise("a) queda menor que 10%"))
print("taxa de acerto por queda de consumo (janela de 12 meses), UCs sem GD:")
display(taxa(insp_uc.filter(~F.col("possui_gd")).withColumn("faixa_queda", faixa_queda), "faixa_queda"))

print("UCs com GD inspecionadas (esperado: 31 inspeções em UCs com GD no total, 1 confirmada):")
display(taxa(insp_uc.withColumn("com_gd", F.col("possui_gd")), "com_gd"))

inspeções com UC na dim_uc: 1.389 de 1.389
taxa de acerto por faixa do score_base (referência: 62,7% no total):


faixa_score,inspecoes,confirmadas,pct_confirmadas
a) 0 a 20,400,35,8.8
b) 20 a 40,77,5,6.5
c) 40 a 60,3,0,0.0
d) 60 a 80,344,311,90.4
e) 80 a 100,565,520,92.0


taxa de acerto por queda de consumo (janela de 12 meses), UCs sem GD:


faixa_queda,inspecoes,confirmadas,pct_confirmadas
a) queda menor que 10%,456,37,8.1
b) queda 10% a 20%,1,0,0.0
d) queda 30% a 40%,20,18,90.0
e) queda 40% a 50%,404,372,92.1
f) queda 50% ou mais,474,441,93.0
z) sem os 12 meses anteriores,3,2,66.7


UCs com GD inspecionadas (esperado: 31 inspeções em UCs com GD no total, 1 confirmada):


com_gd,inspecoes,confirmadas,pct_confirmadas
false,1358,870,64.1
true,31,1,3.2


## 8. Checagens de qualidade

Rodam sobre a fila **antes** de gravar:

| Checagem | Protege |
|---|---|
| Fila não vazia; `id_uc` único; `prioridade` única e contínua de 1 a N | a ordenação |
| `score` entre `SCORE_MIN` e 100; `score` em ordem decrescente de `prioridade` | a regra |
| `queda_explicada_por_gd` ⇒ `kwh_estimado_recuperar_mes = 0` | GD |
| `kwh_estimado_recuperar_mes` >= 0 | "nunca negativo" |
| `tipo_medidor` nos 3 valores; `resultado_ultima_inspecao` nos 2 valores ou nulo | domínios |
| `qtd_inspecoes_anteriores = 0` ⇔ `data_ultima_inspecao` nula | integridade com as inspeções |
| `motivo` preenchido; `municipio`, `bairro` e `classe` preenchidos | o que o Rafael pediu |
| toda UC da fila existe na `gold.dim_uc` | integridade |
| **Soma de `qtd_inspecoes_anteriores` em todas as UCs = 1.389** | o histórico não perdeu nem repetiu inspeções |
| `versao_regra_score` = `v1`; 20 colunas | contrato |

A taxa de acerto por faixa (seção 7) **não é checagem**: é a validação da regra, e mede, não bloqueia.

In [0]:
t = "gold.fila_inspecao"
dq_results.clear()

TIPOS_MEDIDOR = ["ELETRONICO", "ELETROMECANICO", "SMART"]
n_fila = fila.count()

dq_check_valor(t, "fila não vazia", n_fila, n_fila > 0, True)
dq_check_unique(t, "id_uc único", fila, ["id_uc"])
dq_check_unique(t, "prioridade única", fila, ["prioridade"])
# única + mínimo 1 + máximo N = prioridade contínua, sem buracos
dq_check_valor(t, "menor prioridade = 1", n_fila, fila.agg(F.min("prioridade")).first()[0], 1)
dq_check_valor(t, "maior prioridade = número de linhas", n_fila, fila.agg(F.max("prioridade")).first()[0], n_fila)

for coluna in ["id_uc", "prioridade", "score", "motivo", "kwh_estimado_recuperar_mes", "tipo_medidor", "possui_gd", "queda_explicada_por_gd",
               "energia_injetada_atipica", "qtd_inspecoes_anteriores", "municipio", "bairro", "classe", "data_referencia_score", "versao_regra_score"]:
    dq_check(t, f"{coluna} não nulo", fila, F.col(coluna).isNotNull())

dq_check(t, "score entre SCORE_MIN e 100", fila, F.col("score").between(SCORE_MIN, 100))
ordenada = fila.withColumn("score_anterior", F.lag("score").over(Window.orderBy("prioridade")))
dq_check(t, "score em ordem decrescente de prioridade", ordenada, F.col("score_anterior").isNull() | (F.col("score") <= F.col("score_anterior")))
dq_check(t, "queda explicada por GD: energia a recuperar = 0", fila, ~F.col("queda_explicada_por_gd") | (F.col("kwh_estimado_recuperar_mes") == 0))
dq_check(t, "energia a recuperar >= 0", fila, F.col("kwh_estimado_recuperar_mes") >= 0)
dq_check(t, "tipo_medidor nos 3 valores", fila, F.coalesce(F.col("tipo_medidor").isin(*TIPOS_MEDIDOR), F.lit(False)))
dq_check(t, "resultado_ultima_inspecao nos 2 valores ou nulo", fila,
         F.col("resultado_ultima_inspecao").isNull() | F.col("resultado_ultima_inspecao").isin(RESULTADO_CONFIRMADA, RESULTADO_SEM))
dq_check(t, "sem inspeção anterior <=> sem data da última", fila,
         (F.col("qtd_inspecoes_anteriores") == 0) == F.col("data_ultima_inspecao").isNull())
dq_check(t, "motivo não vazio", fila, F.length(F.col("motivo")) > 0)
dq_check(t, "versao_regra_score = v1", fila, F.col("versao_regra_score") == "v1")

# integridade com a dim_uc e com as inspeções
dq_check(t, "toda UC da fila existe na gold.dim_uc", fila.join(dim.select("id_uc", F.lit(True).alias("na_dim")), "id_uc", "left"), F.col("na_dim").isNotNull())
dq_check_valor(t, "soma de qtd_inspecoes_anteriores (todas as UCs) = nº de inspeções da Silver", n_fila,
               pontuada.agg(F.sum("qtd_inspecoes_anteriores")).first()[0], n_insp)
dq_check_valor(t, "20 colunas, como no contrato", n_fila, len(df_fila.columns), 20)

[PASS] gold.fila_inspecao | fila não vazia | obtido True (esperado True)
[PASS] gold.fila_inspecao | id_uc único | 0 chaves duplicadas de 906 linhas
[PASS] gold.fila_inspecao | prioridade única | 0 chaves duplicadas de 906 linhas
[PASS] gold.fila_inspecao | menor prioridade = 1 | obtido 1 (esperado 1)
[PASS] gold.fila_inspecao | maior prioridade = número de linhas | obtido 906 (esperado 906)
[PASS] gold.fila_inspecao | id_uc não nulo | 0/906 linhas falharam
[PASS] gold.fila_inspecao | prioridade não nulo | 0/906 linhas falharam
[PASS] gold.fila_inspecao | score não nulo | 0/906 linhas falharam
[PASS] gold.fila_inspecao | motivo não nulo | 0/906 linhas falharam
[PASS] gold.fila_inspecao | kwh_estimado_recuperar_mes não nulo | 0/906 linhas falharam
[PASS] gold.fila_inspecao | tipo_medidor não nulo | 0/906 linhas falharam
[PASS] gold.fila_inspecao | possui_gd não nulo | 0/906 linhas falharam
[PASS] gold.fila_inspecao | queda_explicada_por_gd não nulo | 0/906 linhas falharam
[PASS] gold.fi

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.fila_inspecao | score em ordem decrescente de prioridade | 0/906 linhas falharam
[PASS] gold.fila_inspecao | queda explicada por GD: energia a recuperar = 0 | 0/906 linhas falharam
[PASS] gold.fila_inspecao | energia a recuperar >= 0 | 0/906 linhas falharam
[PASS] gold.fila_inspecao | tipo_medidor nos 3 valores | 0/906 linhas falharam
[PASS] gold.fila_inspecao | resultado_ultima_inspecao nos 2 valores ou nulo | 0/906 linhas falharam
[PASS] gold.fila_inspecao | sem inspeção anterior <=> sem data da última | 0/906 linhas falharam
[PASS] gold.fila_inspecao | motivo não vazio | 0/906 linhas falharam
[PASS] gold.fila_inspecao | versao_regra_score = v1 | 0/906 linhas falharam
[PASS] gold.fila_inspecao | toda UC da fila existe na gold.dim_uc | 0/906 linhas falharam
[PASS] gold.fila_inspecao | soma de qtd_inspecoes_anteriores (todas as UCs) = nº de inspeções da Silver | obtido 1389 (esperado 1389)
[PASS] gold.fila_inspecao | 20 colunas, como no contrato | obtido 20 (esperado 20)


## 9. Registro das checagens e gravação, só se todas passarem

O resultado das checagens é gravado **sempre** em `silver.dq_log`, mesmo que alguma falhe (`table_name = gold.fila_inspecao`). A tabela só é gravada se **todas** passarem; se alguma falhar, o notebook interrompe com erro, a `_stg` fica para inspeção e a `fila_inspecao` anterior, se existir, permanece. Se todas passarem, a final é gravada a partir da `_stg`, e a `_stg` é apagada.

In [0]:
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_dq_log)

falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada ({tabela_stg} ficou para inspeção):")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    fila.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold)
)
spark.sql(f"DROP TABLE IF EXISTS {tabela_stg}")
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

Todas as 32 checagens passaram. v_energy.gold.fila_inspecao gravada: 906 linhas


## 10. Teste de conexão e leitura da fila

1. **Conexão com a `dim_uc`:** toda UC da fila existe na dimensão, com o mesmo município, bairro e classe.
2. **O topo da fila,** como o Rafael a veria: posição, UC, score, motivo e energia a recuperar.
3. **De que é feita a fila:** por situação (nunca inspecionada, confirmada com consumo que não voltou, última inspeção sem irregularidade) e por bairro.
4. **O que ela diz do diagnóstico:** quantas UCs da fila estão nos quatro bairros e qual a energia a recuperar somada (a ordem de grandeza deve ser comparável a 1,06 e 1,80 milhão de kWh por ano, ou 88 mil e 150 mil por mês).

**Resultado da execução:** **906 UCs**: 32 nunca inspecionadas (score médio 59,7; 3.415 kWh por mês), **831 com irregularidade confirmada antes (85,1; 196.886)** e 43 com a última inspeção sem irregularidade (41,5; 8.581). Por classe: 606 residenciais, 299 comerciais e 1 industrial. **877 das 906 estão nos quatro bairros** (Iputinga 295, Jardim Jordão 282, Prazeres 158 e Macaxeira 142). Energia a recuperar: **208.882 kWh por mês**, 205.591 nos quatro bairros; por classe, 76.675 no residencial e 131.256 no comercial. Anualizados, são ≈ 87% das estimativas do diagnóstico (0,92 e 1,58 milhão de kWh, contra 1,06 e 1,80), como esperado: a fórmula do contrato subestima as UCs com queda recente.

In [0]:
f = spark.table(tabela_gold)

# 1. Conexão com a dim_uc (esperado: 0 e 0)
d = dim.select("id_uc", F.col("municipio").alias("m"), F.col("bairro").alias("b"), F.col("classe").alias("c"))
print("UCs da fila fora da gold.dim_uc:", n_br(f.join(d, "id_uc", "left_anti").count()))
print("UCs com município, bairro ou classe diferente:",
      n_br(f.join(d, "id_uc").filter(~(F.col("municipio").eqNullSafe(F.col("m")) & F.col("bairro").eqNullSafe(F.col("b")) & F.col("classe").eqNullSafe(F.col("c")))).count()))

# 2. O topo, como a tela de perdas mostraria
display(f.orderBy("prioridade").select("prioridade", "id_uc", "score", "motivo", "kwh_estimado_recuperar_mes", "municipio", "bairro", "classe").limit(25))

UCs da fila fora da gold.dim_uc: 0
UCs com município, bairro ou classe diferente: 0


prioridade,id_uc,score,motivo,kwh_estimado_recuperar_mes,municipio,bairro,classe
1,UC101238,100.0,consumo caiu 58% vs. 12 meses anteriores; consumo 68% abaixo dos pares do bairro; medidor eletromecânico com 16 anos; irregularidade confirmada em 2025-12 e o consumo não voltou,175.92,Jaboatão dos Guararapes,Prazeres,COMERCIAL
2,UC102335,100.0,consumo caiu 52% vs. 12 meses anteriores; consumo 61% abaixo dos pares do bairro; medidor eletromecânico com 18 anos; irregularidade confirmada em 2025-09 e o consumo não voltou,171.83,Jaboatão dos Guararapes,Prazeres,COMERCIAL
3,UC105160,100.0,consumo caiu 56% vs. 12 meses anteriores; consumo 67% abaixo dos pares do bairro; medidor eletromecânico com 26 anos; irregularidade confirmada em 2025-12 e o consumo não voltou,169.08,Jaboatão dos Guararapes,Prazeres,COMERCIAL
4,UC100203,100.0,consumo caiu 57% vs. 12 meses anteriores; consumo 66% abaixo dos pares do bairro; medidor eletromecânico com 20 anos; irregularidade confirmada em 2025-08 e o consumo não voltou,148.58,Recife,Macaxeira,COMERCIAL
5,UC124391,100.0,consumo caiu 53% vs. 12 meses anteriores; consumo 67% abaixo dos pares do bairro; medidor eletromecânico com 22 anos; irregularidade confirmada em 2025-12 e o consumo não voltou,127.41,Jaboatão dos Guararapes,Jardim Jordão,COMERCIAL
6,UC139423,100.0,consumo caiu 50% vs. 12 meses anteriores; consumo 70% abaixo dos pares do bairro; medidor eletromecânico com 20 anos; irregularidade confirmada em 2025-09 e o consumo não voltou,114.25,Recife,Iputinga,COMERCIAL
7,UC113436,100.0,consumo caiu 56% vs. 12 meses anteriores; consumo 78% abaixo dos pares do bairro; medidor eletromecânico com 17 anos; irregularidade confirmada em 2025-12 e o consumo não voltou,103.33,Recife,Iputinga,COMERCIAL
8,UC128921,100.0,consumo caiu 47% vs. 12 meses anteriores; consumo 67% abaixo dos pares do bairro; medidor eletromecânico com 25 anos; irregularidade confirmada em 2025-10 e o consumo não voltou,95.42,Jaboatão dos Guararapes,Jardim Jordão,COMERCIAL
9,UC133384,100.0,consumo caiu 50% vs. 12 meses anteriores; consumo 77% abaixo dos pares do bairro; medidor eletromecânico com 16 anos; irregularidade confirmada em 2025-09 e o consumo não voltou,94.34,Jaboatão dos Guararapes,Prazeres,COMERCIAL
10,UC123759,100.0,consumo caiu 47% vs. 12 meses anteriores; consumo 70% abaixo dos pares do bairro; medidor eletromecânico com 15 anos; irregularidade confirmada em 2025-09 e o consumo não voltou,90.50,Jaboatão dos Guararapes,Jardim Jordão,COMERCIAL


In [0]:
# 3. De que é feita a fila
situacao = (F.when(F.col("qtd_inspecoes_anteriores") == 0, "1. nunca inspecionada")
             .when(F.col("resultado_ultima_inspecao") == RESULTADO_CONFIRMADA, "2. confirmada antes (reincidência ou consumo que não voltou)")
             .otherwise("3. última inspeção sem irregularidade (rebaixada)"))
display(f.withColumn("situacao", situacao).groupBy("situacao")
        .agg(F.count("*").alias("ucs"), F.round(F.avg("score"), 1).alias("score_medio"),
             F.round(F.sum("kwh_estimado_recuperar_mes"), 0).alias("kwh_mes_a_recuperar")).orderBy("situacao"))

display(f.groupBy("classe").agg(F.count("*").alias("ucs"), F.round(F.sum("kwh_estimado_recuperar_mes"), 0).alias("kwh_mes_a_recuperar")).orderBy(F.desc("ucs")))
display(f.groupBy("municipio", "bairro").agg(F.count("*").alias("ucs")).orderBy(F.desc("ucs")).limit(12))

# 4. Os quatro bairros do diagnóstico
quatro = [("Recife", "Iputinga"), ("Jaboatão dos Guararapes", "Jardim Jordão"), ("Recife", "Macaxeira"), ("Jaboatão dos Guararapes", "Prazeres")]
cond = F.lit(False)
for m, b in quatro:
    cond = cond | ((F.col("municipio") == m) & (F.col("bairro") == b))
tot = f.count()
q = f.filter(cond)
print(f"UCs da fila nos quatro bairros: {n_br(q.count())} de {n_br(tot)}")            # esperado: a maior parte da fila
print("energia a recuperar (kWh por mês) na fila:", f.agg(F.round(F.sum("kwh_estimado_recuperar_mes"), 0)).first()[0],
      "| nos quatro bairros:", q.agg(F.round(F.sum("kwh_estimado_recuperar_mes"), 0)).first()[0])

situacao,ucs,score_medio,kwh_mes_a_recuperar
1. nunca inspecionada,32,59.7,3415
2. confirmada antes (reincidência ou consumo que não voltou),831,85.1,196886
3. última inspeção sem irregularidade (rebaixada),43,41.5,8581


classe,ucs,kwh_mes_a_recuperar
RESIDENCIAL,606,76675
COMERCIAL,299,131256
INDUSTRIAL,1,951


municipio,bairro,ucs
Recife,Iputinga,295
Jaboatão dos Guararapes,Jardim Jordão,282
Jaboatão dos Guararapes,Prazeres,158
Recife,Macaxeira,142
Recife,Brasília Teimosa,3
Cabo de Santo Agostinho,Pirapama,2
Recife,Coelhos,2
Igarassu,Saramandaia,2
Olinda,Aguazinha,2
Paulista,Vila Torres Galvão,2


UCs da fila nos quatro bairros: 877 de 906
energia a recuperar (kWh por mês) na fila: 208882 | nos quatro bairros: 205591


## 11. As 32 UCs que nunca foram inspecionadas

São as candidatas a **descoberta nova**. A fila as coloca longe do topo: a primeira nunca inspecionada aparece na **prioridade 243**, e as demais, a partir da **824**, porque as revisitas (831 UCs com irregularidade confirmada e consumo que não voltou) têm score maior. As revisitas concentram **94% da energia a recuperar** (196.886 de 208.882 kWh por mês); as 32 nunca inspecionadas, 3.415.

**Resultado da execução:**
- **Só 3 estão nos quatro bairros do diagnóstico** (Iputinga, Macaxeira e Prazeres, todas residenciais, com queda de 26% a 43%): são as sobras do grupo.
- **As outras 29 estão espalhadas por 7 municípios** (Recife, Cabo de Santo Agostinho, Olinda, Paulista, Igarassu, Araçoiaba e Jaboatão): UCs isoladas, com queda de 19% a 46% e score de 40 a 76. No total das 32: 28 residenciais, 3 comerciais e 1 industrial; 7 com medidor eletromecânico, 6 `SMART` e 19 eletrônicos.
- **Seis consomem mais que os pares** (por exemplo, a UC123380, em Paulista, caiu 46% e consome 72% acima), e a queda sozinha já dá 60 pontos. O perfil é diferente do grupo.

**Este é o limite da validação:** o sinal de queda de consumo foi validado **apenas** nas UCs dos quatro bairros (91% a 93% de acerto). Fora deles não há inspeções de UCs com queda forte, então **a taxa de acerto do sinal fora do grupo é desconhecida**. As 29 UCs dispersas são o melhor teste da regra com dados novos: inspecionar algumas e registrar o resultado dá a primeira validação independente da `v1`.

In [0]:
f = spark.table(tabela_gold)
display(f.filter("qtd_inspecoes_anteriores = 0").orderBy("prioridade")
        .select("prioridade", "id_uc", "score", "queda_consumo_pct", "consumo_vs_pares_pct", "tipo_medidor",
                "possui_gd", "classe", "municipio", "bairro", "kwh_estimado_recuperar_mes"))
# esperado: 32 linhas; prioridade 243 (UC138519, Iputinga) e as demais a partir da 824

prioridade,id_uc,score,queda_consumo_pct,consumo_vs_pares_pct,tipo_medidor,possui_gd,classe,municipio,bairro,kwh_estimado_recuperar_mes
243,UC138519,85.2,35.37,-58.44,ELETROMECANICO,false,RESIDENCIAL,Recife,Iputinga,31.41
824,UC137016,76.4,43.93,-50.37,ELETRONICO,false,RESIDENCIAL,Cabo de Santo Agostinho,Pirapama,62.08
825,UC140972,76.1,39.23,-53.68,ELETRONICO,false,RESIDENCIAL,Cabo de Santo Agostinho,Zona Industrial de Suape,48.08
826,UC106923,74.8,44.14,-46.24,ELETRONICO,false,RESIDENCIAL,Recife,Brejo da Guabiraba,73.41
827,UC143593,74.5,40.41,-58.57,SMART,false,RESIDENCIAL,Recife,Barro,48.08
828,UC122703,74.2,33.82,-37.45,ELETROMECANICO,false,RESIDENCIAL,Recife,Brasília Teimosa,52.00
831,UC115926,70.8,35.38,-6.01,ELETROMECANICO,false,RESIDENCIAL,Cabo de Santo Agostinho,Pirapama,82.17
832,UC104240,67.9,32.83,-39.20,ELETROMECANICO,false,RESIDENCIAL,Olinda,Aguazinha,48.50
833,UC122876,67.4,38.25,-49.17,SMART,false,RESIDENCIAL,Olinda,Casa Caiada,49.08
834,UC119489,65.4,35.19,-68.94,ELETRONICO,false,RESIDENCIAL,Recife,Ilha Joana Bezerra,24.17
